### Lecture 7: Let's build GPT: from scratch, in code, spelled out

[Youtube video lecture](https://www.youtube.com/watch?v=kCc8FmEb1nY)  
[Github repo (for this video)](https://github.com/karpathy/ng-video-lecture)  
[nanoGPT repo](https://github.com/karpathy/nanoGPT)

Video upload date: Jan 18, 2023 (length: 1 hr 56 min)  
Watched on: Nov 26, 2025  
Reproduced on: Nov 28 and 30, 2025  

Content:
- a transformer-based character language model 

In [1]:
import torch
import torch.nn as nn
from torch.nn import functional as F

def get_device():
    if torch.cuda.is_available():
        gpu_id = 3
        device = torch.device(f"cuda:{gpu_id}") 
    elif torch.backends.mps.is_available():
        device = torch.device("mps")
    else:
        device = torch.device("cpu")
    # print(device.type)
    print(device)
    return device

In [2]:
# hyperparameters
batch_size = 64  # number of independent sequences that we process in parallel
block_size = 256  # number of maximum context length for next character prediction
max_iters = 5000
eval_interval = 500
learning_rate = 3e-4 
eval_iters = 200
n_embd = 384
n_head = 6
n_layer = 6
dropout = 0.2

torch.manual_seed(1337)

device = get_device()

cuda:3


In [3]:
# data processing
with open('data/tinyshakespeare.txt', 'r', encoding='utf-8') as f:
    text = f.read()
print(len(text))

# here are all the unique characters that occur in this text
chars = sorted(list(set(text)))
vocab_size = len(chars)
# create a mapping from characters to integers
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string

# Train and test splits
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9*len(data)) # first 90% will be train, rest val
train_data = data[:n]
val_data = data[n:]

# data loading
def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    ### NOTE: predict next character for every position in the block in parallel 
    ### (in previous lessons, we conly do it for the last position)
    ### In other words, each position in the sequence is its own 
    ###    training example for “next token given all previous ones”.
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

1115394


In [4]:
@torch.no_grad()
def estimate_loss():
    '''regularly evaluate on the train/val splits'''
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

In [5]:
# Attention!
class Head(nn.Module):
    '''one head of self-attention'''
    
    def __init__(self, head_size):
        super().__init__()
        # head_size is the dimension of output vector/embedidng for each character
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # input of size (batch, time-step, channels)
        # output of size (batch, time-step, head size) 
        B, T, C = x.shape
        k = self.key(x)    # (B, T, hs)
        q = self.query(x)  # (B, T, hs)
        # compute attention scores ("affinities")
        wei = q @ k.transpose(-2, -1) * k.shape[-1]**(-0.5) # (B, T, hs) @ (B, hs, T) -> (B, T, T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf')) # (B, T, T)
        wei = F.softmax(wei, dim=-1) # (B, T, T)
        wei = self.dropout(wei)
        # perform the weighted  aggregation of the values
        v = self.value(x)  # (B, T, hs)
        out = wei @ v  # (B, T, T) @ (B, T, hs) -> (B, T, hs)
        return out

class MultiHeadAttention(nn.Module):
    '''multiple heads of self-attention in parallel'''

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(head_size * num_heads, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1) # concatenate the channel/vector
        out = self.dropout(self.proj(out))
        return out

class FeedForward(nn.Module):
    '''a simple linear layer followed by a non-linearity'''

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd), # change 4 to another number if you like
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout)
        )
    
    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    '''tranformer block: communication followed by computation'''

    def __init__(self, n_embd, n_head):
        # n_embd: embedding dimension
        # n_head: the number of heads we'd like
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)
        # the basic unit for LayerNorm is the embedding of a single chacacter
        # the basic unit for BatchNorm is one single dimension of embedding from all characters from a batch
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        # Notice: pre-attention LN here, different from original attention paper
        x = x + self.sa(self.ln1(x))  # also, residual connection here
        x = x + self.ffwd(self.ln2(x))
        return x

class GPTLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)  # final layer norm
        self.lm_head = nn.Linear(n_embd, vocab_size)

        self.apply(self.__init__weights)

    def __init__weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
    
    def forward(self, idx, targets=None):
        B, T = idx.shape

        # idx and targets are both (B, T) tensor of integers
        tok_emb = self.token_embedding_table(idx)  # (B, T, C)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device))  # (T, C)
        x = tok_emb + pos_emb  # (B, T, C)
        x = self.blocks(x)  # (B, T, C)
        x = self.ln_f(x)  # (B, T, C)
        logits = self.lm_head(x)  # (B, T, vocab_size)

        if targets is None:  # for inference
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T) 
            # a B*T separate classification problems, one per position
            ### So we’re not predicting a single token; 
            ###    we’re predicting the next token at every time step in the block
            loss = F.cross_entropy(logits, targets)

        return logits, loss
    
    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # crop idx to the last block_size tokens
            idx_cond = idx[:, -block_size:]
            # get the predictions
            logits, loss = self(idx_cond)
            ### during generation, we focus only on the last time step
            ### use the logits from last position to predict next character
            logits = logits[:, -1, :]  # become (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1)  # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1)  # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1)  # (B, T+1)
        return idx

In [6]:
model = GPTLanguageModel().to(device)
# m = model.to(device)
# print the number of parameters in the model
print(sum(p.numel() for p in model.parameters())/1e6, 'M parameters')

10.788929 M parameters


In [7]:
# create a Pytorch optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

In [8]:
for iter in range(max_iters):

    # every once in a while evaluate the loss on train and val sets
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")
    
    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

step 0: train loss 4.2221, val loss 4.2306
step 500: train loss 1.7570, val loss 1.9124
step 1000: train loss 1.3902, val loss 1.6033
step 1500: train loss 1.2664, val loss 1.5218
step 2000: train loss 1.1852, val loss 1.4953
step 2500: train loss 1.1249, val loss 1.4928
step 3000: train loss 1.0695, val loss 1.4868
step 3500: train loss 1.0187, val loss 1.4997
step 4000: train loss 0.9621, val loss 1.5073
step 4500: train loss 0.9111, val loss 1.5357
step 4999: train loss 0.8577, val loss 1.5591


In [9]:
# generate from the model
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(model.generate(context, max_new_tokens=500)[0].tolist()))


Had you to do up, the your honour I please; you know
'll ven me yet shall this tus up a dowry to him. To him this
very
'Tward his necks; his punit is honour with his
disposing. 'Tis no true: thou wilt but make
down, Wome laugh for him; though this, and man
but a streets, to the Tarpince offinit
her mis-headed her whorre, thought his eyes or notticly,
and stabb'd with him, and with all hope,
ave our own kttteres.

First Senator:
Marcius, are odds:
The man Ediciply of thing gone:
The duke the nurs
